# parallel per-plane writer demo

compares the standard `mbo.imread` -> `mbo.imwrite` path (sequential, `fix_phase=False`) against a per-plane `ProcessPoolExecutor` that mirrors the ScanImage reader's page-index iteration.

each parallel worker opens its own `TiffFile` handles and calls `tf.asarray(key=...)` on a disjoint set of page indices, so no two workers ever touch the same page.

In [7]:
from pathlib import Path
import time
import shutil

SRC      = Path('D:/demo/raw')
DST_SEQ  = Path('D:/demo/par_io/default')
DST_PAR  = Path('D:/demo/par_io/parallel')

for d in (DST_SEQ, DST_PAR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

## 1. sequential baseline

`mbo.imread` -> `mbo.imwrite` with `fix_phase=False`. one process, one plane at a time.

In [8]:
import mbo_utilities as mbo

arr = mbo.imread(SRC)
if hasattr(arr, 'fix_phase'):
    arr.fix_phase = False

t0 = time.perf_counter()
mbo.imwrite(arr, DST_SEQ, ext='.tif', overwrite=True)
t_seq = time.perf_counter() - t0
print(f'sequential: {t_seq:.2f}s')

Counting frames:   0%|          | 0/2 [00:00<?, ?it/s]

Writing TIFF:   0%|          | 0/22036 [00:00<?, ?pg/s]

sequential: 15.25s


## 2. parallel per-plane

page formula mirrors `ScanImageArray._read_pages` (`tiff.py:1253-1264`):
```
page_in_file = t * num_channels + z
```
for LBM data `num_channels == nz` (z-planes stored in ScanImage's channel slots).

each worker handles ONE plane, walks every source file in order, and reads `[t*nz + z for t in range(frames_in_file)]` from each. plane `z=0` reads pages `[0, nz, 2*nz, ...]`; plane `z=1` reads `[1, nz+1, 2*nz+1, ...]`. fully disjoint — no shared file handles, no overlapping page indices.

**windows note:** `ProcessPoolExecutor` uses the `spawn` start method on windows. workers re-import the function by module path, so the worker has to live in a real `.py` file — a function defined inside the notebook lives in `__main__` of an interactive kernel and can't be re-imported, which causes `BrokenProcessPool`. the next cell writes the worker to a sibling module so children can import it.

In [9]:
%%writefile _parallel_io_worker.py
from pathlib import Path
import numpy as np
import tifffile


def _stitch_rois(raw, layout):
    """mirror ScanImageArray.process_rois (tiff.py:1474). raw: (T, H_page, W_page)."""
    if layout is None or len(layout['src_yslices']) <= 1:
        return raw
    T = raw.shape[0]
    out = np.zeros((T, layout['max_height'], layout['total_width']), dtype=raw.dtype)
    for (sy0, sy1), (oy0, oy1), (ox0, ox1) in zip(
        layout['src_yslices'], layout['out_yslices'], layout['out_xslices']
    ):
        out[:, oy0:oy1, ox0:ox1] = raw[:, sy0:sy1, :]
    return out


def write_plane(job):
    """runs in a fresh process. opens its own TiffFile handles."""
    src_files, frames_per_file, num_channels, z_idx, roi_layout, out_path = job

    chunks = []
    pages_touched = []
    for path, n_frames in zip(src_files, frames_per_file):
        page_indices = [t * num_channels + z_idx for t in range(n_frames)]
        pages_touched.append((Path(path).name, page_indices[0], page_indices[-1]))
        with tifffile.TiffFile(path) as tf:
            raw = tf.asarray(key=page_indices)
        if raw.ndim == 2:
            raw = raw[np.newaxis, ...]
        chunks.append(_stitch_rois(raw, roi_layout))

    full = np.concatenate(chunks, axis=0)
    tifffile.imwrite(str(out_path), full, bigtiff=True)
    return z_idx, full.shape, pages_touched


Overwriting _parallel_io_worker.py


In [10]:
# discover source layout once on the parent so workers don't redo it.
import tifffile

if SRC.is_dir():
    src_files = sorted(str(p) for p in SRC.glob('*.tif*'))
else:
    src_files = [str(SRC)]

nz = int(getattr(arr, 'num_channels', None) or arr.shape[-3])
frames_per_file = getattr(arr, '_frames_per_file', None)
if frames_per_file is None:
    frames_per_file = []
    for p in src_files:
        with tifffile.TiffFile(p) as tf:
            frames_per_file.append(len(tf.pages) // nz)

# extract roi stitching layout (mirrors ScanImageArray.process_rois inputs).
# convert slice objects to (start, stop) tuples so they pickle cleanly to workers.
roi_layout = None
if hasattr(arr, '_rois') and arr._rois:
    rois = arr._rois
    src_y = arr.yslices            # vertical strip in raw page per roi
    out_y = arr.output_yslices     # destination y in stitched canvas
    out_x = arr.output_xslices     # destination x (offset by previous widths)
    roi_layout = {
        'src_yslices': [(s.start or 0, s.stop) for s in src_y],
        'out_yslices': [(s.start or 0, s.stop) for s in out_y],
        'out_xslices': [(s.start or 0, s.stop) for s in out_x],
        'max_height':  max(r['height'] for r in rois),
        'total_width': sum(r['width']  for r in rois),
    }

print(f'{len(src_files)} files, nz={nz}, frames_per_file={frames_per_file[:3]}{"..." if len(frames_per_file) > 3 else ""}')
if roi_layout:
    print(f'rois: {len(roi_layout["src_yslices"])}, stitched frame: '
          f'({roi_layout["max_height"]}, {roi_layout["total_width"]})')
else:
    print('rois: 1 (no stitching)')


2 files, nz=14, frames_per_file=[701, 873]
rois: 2, stitched frame: (550, 448)


In [11]:
import sys
from concurrent.futures import ProcessPoolExecutor

# make sure the notebook's cwd is on sys.path so the children can import the worker module.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from _parallel_io_worker import write_plane

jobs = [
    (src_files, frames_per_file, nz, z, roi_layout, DST_PAR / f'plane_{z:02d}.tif')
    for z in range(nz)
]

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=min(nz, 30)) as pool:
    for z_idx, shape, pages in pool.map(write_plane, jobs):
        first_file, first_pi, last_pi = pages[0]
        print(f'plane {z_idx:02d}: shape={shape} pages [{first_pi}..{last_pi}] (stride {nz}) in {first_file}')
t_par = time.perf_counter() - t0

print(f'\nsequential: {t_seq:.2f}s')
print(f'parallel:   {t_par:.2f}s ({t_seq / t_par:.1f}x speedup)')


plane 00: shape=(1574, 550, 448) pages [0..9800] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 01: shape=(1574, 550, 448) pages [1..9801] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 02: shape=(1574, 550, 448) pages [2..9802] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 03: shape=(1574, 550, 448) pages [3..9803] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 04: shape=(1574, 550, 448) pages [4..9804] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 05: shape=(1574, 550, 448) pages [5..9805] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 06: shape=(1574, 550, 448) pages [6..9806] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 07: shape=(1574, 550, 448) pages [7..9807] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 08: shape=(1574, 550, 448) pages [8..9808] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 09: shape=(1574, 550, 448) pages [9..9809] (stride 14) in demo_mk355_7_27_2025_00001.tif
plane 10: shape=(1574, 550, 448) pages [10..9810] 

Single hemisphere

In [12]:
from pathlib import Path
import time
import shutil
import mbo_utilities as mbo

In [ ]:
SRC      = Path('D:/jeff/raw')
DST_SEQ  = Path('D:/jeff/par_io/default')
DST_PAR  = Path('D:/jeff/par_io/parallel')

for d in (DST_SEQ, DST_PAR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

arr = mbo.imread(SRC)
if hasattr(arr, 'fix_phase'):
    arr.fix_phase = False

t0 = time.perf_counter()
mbo.imwrite(arr, DST_SEQ, ext='.tif', overwrite=True)
t_seq = time.perf_counter() - t0
print(f'sequential: {t_seq:.2f}s')

Counting frames:   0%|          | 0/3 [00:00<?, ?it/s]

Writing TIFF:   0%|          | 0/35280 [00:00<?, ?pg/s]

In [ ]:
if SRC.is_dir():
    src_files = sorted(str(p) for p in SRC.glob('*.tif*'))
else:
    src_files = [str(SRC)]

nz = int(getattr(arr, 'num_channels', None) or arr.shape[-3])
frames_per_file = getattr(arr, '_frames_per_file', None)
if frames_per_file is None:
    frames_per_file = []
    for p in src_files:
        with tifffile.TiffFile(p) as tf:
            frames_per_file.append(len(tf.pages) // nz)

# extract roi stitching layout (mirrors ScanImageArray.process_rois inputs).
# convert slice objects to (start, stop) tuples so they pickle cleanly to workers.
roi_layout = None
if hasattr(arr, '_rois') and arr._rois:
    rois = arr._rois
    src_y = arr.yslices            # vertical strip in raw page per roi
    out_y = arr.output_yslices     # destination y in stitched canvas
    out_x = arr.output_xslices     # destination x (offset by previous widths)
    roi_layout = {
        'src_yslices': [(s.start or 0, s.stop) for s in src_y],
        'out_yslices': [(s.start or 0, s.stop) for s in out_y],
        'out_xslices': [(s.start or 0, s.stop) for s in out_x],
        'max_height':  max(r['height'] for r in rois),
        'total_width': sum(r['width']  for r in rois),
    }

print(f'{len(src_files)} files, nz={nz}, frames_per_file={frames_per_file[:3]}{"..." if len(frames_per_file) > 3 else ""}')
if roi_layout:
    print(f'rois: {len(roi_layout["src_yslices"])}, stitched frame: '
          f'({roi_layout["max_height"]}, {roi_layout["total_width"]})')
else:
    print('rois: 1 (no stitching)')
